# Assignment 1: Collecting the Data

In the **first module** we met the main libraries for working with vector spatial data, learned to read and write data in different formats, and exported features from OpenStreetMap.

In this assignment you will prepare the data for the course project — an analysis of how accessible everyday amenities are, in the spirit of the **15-minute city**.

Along the way we will:

- choose a study area;
- download everyday amenities from OSM by category;
- clean and prepare the data;
- save the result for the analysis that follows.

> The code cells below are placeholders for your own work: they are left unexecuted on purpose. Fill them in as you go, and run them in your own copy of the notebook.


## How the Project Data Is Organised

Every assignment in this course feeds the next one, so it pays to agree on where the data lives before writing any code.

Keep everything in **one GeoPackage**, `project.gpkg`, with one layer per dataset:

| Layer | Content | Created in |
| --- | --- | --- |
| `area` | boundary of the study area | assignment 1 |
| `poi_<category>` | everyday amenities, one layer per category (`poi_grocery`, `poi_pharmacy`, …) | assignment 1 |
| `landuse` | land use polygons | assignment 2 |
| `stops` | public transport stops | assignment 3 |
| `buildings` | buildings with their accessibility flags | assignment 3 |
| `stops_catchment` | catchment areas around the stops | assignment 3 |
| `iso_<category>` | walking catchment areas, one row per time interval | assignment 4 |

Two rules that save a lot of debugging later:

- **store everything in EPSG:4326** and reproject when you calculate or display, rather than storing layers in different systems;
- **keep the category names stable** — `grocery` in assignment 1 must stay `grocery` in assignments 4, 5 and 6.

A GeoPackage keeps each layer's CRS and attributes intact, so this one file carries the whole project.

## Suggested Steps


### Step 0. Importing Libraries

Import everything you will need. One import is there to get you started — add the rest as you go.


In [ ]:
import osmnx as ox

# cache OSM responses on disk, so repeating a query does not hit the server again
ox.settings.cache_folder = "../../cache"

# add the other libraries here


### Step 1. Choosing a Study Area


Pick the area you will work with. To start, take a **district of a large city** or a **small town** (up to about 150,000 residents): the data stays light and downloads and processing stay fast — which matters while you are still experimenting with the sequence of steps.

Later you can apply the same workflow to a bigger area and compare the results.


#### 1.1. Load the Boundary from OSM

`geocode_to_gdf()` geocodes a place name and returns its boundary as a `GeoDataFrame` — see [Exporting Data from OSM](spData_4.ipynb), section 1.


In [ ]:
area_name = ""  # the name of your district or town

area = ox.geocode_to_gdf(area_name)


#### 1.2. Exploring the Data

The checks below are the ones from [Exploring a Dataset](spData_3.ipynb) — run them on your own layer.


##### 1.2.1. Visualisation

Look at the area on a map. Is it the district or town you had in mind?


In [ ]:
# your code


##### 1.2.2. Attributes

See which attributes the table holds and what data types they have.


In [ ]:
# your code


##### 1.2.3. Geometry

Check which geometry types are present in the data.


In [ ]:
# your code


#### 1.3. Preparing the Data


##### 1.3.1. Selecting Attributes (optional)

Decide which attribute fields are worth keeping and drop the rest.

You **must keep the geometry column** (`geometry`), and we recommend keeping the OSM identifier as well. In OSMnx the identifier is not a regular column: it lives in the index, so extract it into a column of its own first — as we did in [Exporting Data from OSM](spData_4.ipynb), section 4.3:

```python
area["osm_id"] = area.index.get_level_values("id")
```


In [ ]:
# your code


##### 1.3.2. Filtering by Geometry Type (optional)

Are there features with a geometry other than `Polygon` / `MultiPolygon`?

If so, drop them and keep only polygons and multipolygons.


In [ ]:
# your code


#### 1.4. Saving

Save the prepared boundary as the `area` layer of `project.gpkg` — see [Reading Vector Data](spData_2.ipynb), section 2.3, for how to write a layer into a GeoPackage.

In [ ]:
# your code

# area.to_file("project.gpkg", layer="area", driver="GPKG")

### Step 2. Everyday Amenities


#### 2.1. Choosing the Categories

Choose **at least four types of everyday amenities** to assess accessibility for.

For example:

- grocery shops
- pharmacies
- schools and kindergartens
- clinics
- parks
- libraries or cultural venues

_Public transport stops are deliberately not on this list — they are the subject of [assignment 3](../module_3/geoprocessing_task.ipynb), where we look at them separately._

For each type:

1. Find the matching **OSM tags** (there may be several).
2. Collect the categories into a single dictionary, where the key is a short category name and the value is the tag dictionary.

Keep in mind that one type of amenity may correspond to several values of the same key (`amenity`, `shop`, `leisure` and others), and sometimes to different keys altogether. The full list is in the OSM [Map Features](https://wiki.openstreetmap.org/wiki/Map_features) documentation.

The dictionary is worth the small extra effort: from here on, every step that has to be repeated for each category — downloading, cleaning, building catchment areas, counting population — becomes a loop over `categories` instead of copied-out code. Assignments 4, 5 and 6 all build on it.

In [ ]:
# example

categories = {
    "grocery": {
        "shop": ["supermarket", "convenience", "grocery"],
        "amenity": ["marketplace"]
    },
    "pharmacy": {
        "amenity": ["pharmacy"]
    },
    # add your other categories here
}

#### 2.2. Downloading the Data from OSM

Download the features for every category. With the dictionary in place this is one loop, and the results can be kept in a dictionary of the same shape.

In [ ]:
# example

osm_data = {}

for category_name, tags in categories.items():
    osm_data[category_name] = ox.features_from_place(area_name, tags)

#### 2.3. Exploring the Data

For each dataset:

1. Look at the data on a map.
2. Count the features.
3. Inspect the attributes.
4. Check the geometry types.

If a category comes back with very few features — or none at all — it is usually the tags rather than the city: go back to step 2.1 and check them against the OSM documentation.

If the tags are right and the category is still nearly empty, the area itself is probably mapped too sparsely in OSM. It is much cheaper to pick a different study area now than after five more assignments have been built on it.

In [ ]:
# your code


#### 2.4. Preparing the Data

Clean up each dataset.

1. Keep a single geometry type. Amenities in OSM are mapped both as points and as building outlines, so decide which representation you want — for example keep only `Point` features, or keep only `Polygon` / `MultiPolygon` ones — depending on how completely your area is mapped.

2. Drop the attribute columns you do not need, keeping the ones that will be useful later: `geometry`, the OSM identifier, `name`, and the field that carries the amenity type (`amenity`, `shop`, `leisure`, and so on).


In [ ]:
# your code


#### 2.5. Saving the Data

Save the prepared datasets into the same `project.gpkg`, one layer per category, following the naming agreed at the top of the notebook: `poi_grocery`, `poi_pharmacy`, and so on.

Then check that all the layers really are in the one file — read it back and list the layers with `gpd.list_layers()`, as in [Reading Vector Data](spData_2.ipynb), section 1.3.

In [ ]:
# your code

# for category_name, gdf in osm_data.items():
#     gdf.to_file("project.gpkg", layer=f"poi_{category_name}", driver="GPKG")

## What You Should Have

You should now have `project.gpkg` with clean data covering your study area: the `area` layer with its boundary, plus a `poi_<category>` layer for every category of everyday amenities — and a `categories` dictionary that names them. These are the datasets the next stages of the analysis build on.